# AT1: Perceptron e KNN em Prática
**Disciplina:** Aprendizagem de Máquina  
**Alunox:** Bruno de lucca   
**Objetivo:** Implementação vetorizada dos algoritmos Perceptron, KNN e Sistema de Recomendação por Similaridade Espacial usando NumPy.

## Desafio 1: Triagem de Fraude com Perceptron Treinável
Classificação de transações financeiras com base em:
- $x_1$: Valor da Transação Normalizado (0.0 a 10.0)
- $x_2$: Frequência de Operações Recentes (últimos 5 min)

**Classes:** `0` (Legítima) | `1` (Suspeita de Fraude)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# 1. Dados de Treinamento
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# 2. Amostras de Teste
transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

# 3. Função de Treinamento (Algoritmo de Rosenblatt)
def treinar_perceptron(X, y, lr=0.1, epocas=100):
   
    n_amostras, n_atributos = X.shape
    w = np.ones(n_atributos) 
    b = 1.0                  
    
    for _ in range(epocas):
        erros_acumulados = 0
        for i in range(n_amostras):
            z = np.dot(X[i], w) + b
            
            y_pred = 1 if z >= 0 else 0
            
            erro = y[i] - y_pred
            
            if erro != 0:
                w += lr * erro * X[i]
                b += lr * erro
                erros_acumulados += 1
                
        if erros_acumulados == 0:
            break
            
    return w, b

# 4. Função de Inferência
def predizer_perceptron(X_amostra, w, b):
    z = np.dot(X_amostra, w) + b
    return 1 if z >= 0 else 0

# 5. Execução do Treinamento
w_calibrado, b_calibrado = treinar_perceptron(X_treino_fraude, y_treino_fraude, lr=0.1, epocas=100)

print("="*60)
print("DESAFIO 1: RESULTADOS DO TREINAMENTO E INFERÊNCIA DO PERCEPTRON")
print("="*60)
print(f"Pesos Finais (w): {w_calibrado}")
print(f"Viés Final (b):   {b_calibrado:.4f}\n")

# 6. Avaliação das Amostras de Teste
rotulos_map = {0: "Legítima", 1: "Suspeita (Risco de Fraude)"}

pred_A = predizer_perceptron(transacao_A, w_calibrado, b_calibrado)
pred_B = predizer_perceptron(transacao_B, w_calibrado, b_calibrado)

print(f"Transação A {transacao_A.tolist()} -> Diagnóstico: Classe {pred_A} ({rotulos_map[pred_A]})")
print(f"Transação B {transacao_B.tolist()} -> Diagnóstico: Classe {pred_B} ({rotulos_map[pred_B]})")

DESAFIO 1: RESULTADOS DO TREINAMENTO E INFERÊNCIA DO PERCEPTRON
Pesos Finais (w): [-5.55111512e-16  4.00000000e-01]
Viés Final (b):   -1.2000

Transação A [5.0, 4.0] -> Diagnóstico: Classe 1 (Suspeita (Risco de Fraude))
Transação B [7.0, 6.5] -> Diagnóstico: Classe 1 (Suspeita (Risco de Fraude))


## Desafio 2: Predição de Risco de Churn com KNN
Identificação de clientes corporativos sob risco de cancelamento com base em:
- $x_1$: Dias de Inatividade
- $x_2$: Chamados Críticos em Aberto

**Classes:** `0` (Baixo Risco) | `1` (Alto Risco)

In [13]:
# 1. Dados de Treinamento
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# 2. Clientes de Teste
cliente_1 = np.array([13.0, 10.0]) # K = 3, Manhattan
cliente_2 = np.array([7.0, 1.5])   # K = 5, Euclidiana

# 3. Função de Classificação KNN
def classificar_knn(X_treino, y_treino, ponto_consulta, k=3, metrica='euclidiana'):
    
    # Cálculo vetorizado das distâncias sem laços manuais
    if metrica.lower() == 'euclidiana':
        distancias = np.sqrt(np.sum((X_treino - ponto_consulta) ** 2, axis=1))
    elif metrica.lower() == 'manhattan':
        distancias = np.sum(np.abs(X_treino - ponto_consulta), axis=1)
    else:
        raise ValueError("Métrica não suportada. Escolha 'euclidiana' ou 'manhattan'.")
    
    # Índices dos K menores valores de distância
    indices_vizinhos = np.argsort(distancias)[:k]
    
    # Classes dos vizinhos e contagem por maioria de votos
    classes_vizinhos = y_treino[indices_vizinhos]
    distancias_vizinhos = distancias[indices_vizinhos]
    
    # Apuração por voto majoritário
    bincount = np.bincount(classes_vizinhos)
    classe_predita = np.argmax(bincount)
    
    return classe_predita, indices_vizinhos, distancias_vizinhos

# 4. Execução para os Clientes de Teste
rotulos_churn = {0: "Baixo Risco", 1: "Alto Risco"}

def diagnosticar_cliente(nome, cliente, k, metrica):
    classe, idx_viz, dist_viz = classificar_knn(X_treino_churn, y_treino_churn, cliente, k=k, metrica=metrica)
    print(f"--- Diagnóstico para {nome} {cliente.tolist()} ---")
    print(f"Configuração: K = {k} | Métrica = {metrica.capitalize()}")
    print(f"Classe Predita: {classe} ({rotulos_churn[classe]})")
    print(f"Índices dos Vizinhos na Base: {idx_viz.tolist()}")
    print(f"Distâncias para os Vizinhos:  {np.round(dist_viz, 4).tolist()}\n")

print("="*60)
print("DESAFIO 2: DIAGNÓSTICO DE RISCO DE CHURN COM KNN")
print("="*60)
diagnosticar_cliente("Cliente 1", cliente_1, k=3, metrica='manhattan')
diagnosticar_cliente("Cliente 2", cliente_2, k=5, metrica='euclidiana')

DESAFIO 2: DIAGNÓSTICO DE RISCO DE CHURN COM KNN
--- Diagnóstico para Cliente 1 [13.0, 10.0] ---
Configuração: K = 3 | Métrica = Manhattan
Classe Predita: 1 (Alto Risco)
Índices dos Vizinhos na Base: [4, 5, 7]
Distâncias para os Vizinhos:  [8.0, 10.0, 13.0]

--- Diagnóstico para Cliente 2 [7.0, 1.5] ---
Configuração: K = 5 | Métrica = Euclidiana
Classe Predita: 0 (Baixo Risco)
Índices dos Vizinhos na Base: [3, 2, 1, 0, 4]
Distâncias para os Vizinhos:  [1.118, 2.0616, 4.272, 5.0249, 8.3815]



## Desafio 3: Recomendação de Servidores Cloud por Similaridade Espacial
Dimensionamento automático de Máquinas Virtuais no espaço 3D:
- Atributos: `vCPUs`, `Memória RAM (GB)`, `Armazenamento SSD NVMe (GB)`

In [ ]:
# 1. Catálogo e Nomes dos Servidores
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],    
    [4.0, 8.0, 100.0],   
    [8.0, 16.0, 250.0],  
    [16.0, 32.0, 500.0], 
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

# 2. Entrada de Teste (Requisitos da Aplicação: 12 vCPUs, 28 GB RAM, 450 GB SSD)
demanda_aplicacao = np.array([12.0, 28.0, 450.0]) 
K_recomendacoes = 2

# 3. Função de Recomendação por Similaridade Espacial
def recomendar_servidores(catalogo, nomes, demanda, k=2):
    
    # Cálculo vetorizado da distância euclidiana em 3D
    distancias = np.sqrt(np.sum((catalogo - demanda) ** 2, axis=1))
    
    # Ordenação dos índices da menor para a maior distância
    indices_ordenados = np.argsort(distancias)[:k]
    
    recomendacoes = []
    for pos, idx in enumerate(indices_ordenados, start=1):
        recomendacoes.append({
            "posicao": pos,
            "nome": nomes[idx],
            "specs": catalogo[idx],
            "distancia": distancias[idx]
        })
        
    return recomendacoes

# 4. Execução e Exibição do Relatório
resultados_rec = recomendar_servidores(catalogo_servidores, nomes_servidores, demanda_aplicacao, k=K_recomendacoes)

print("="*65)
print("DESAFIO 3: RELATÓRIO DE RECOMENDAÇÃO DE SERVIDORES CLOUD")
print("="*65)
print(f"Perfil Demanda Solicitada: {demanda_aplicacao[0]:.0f} vCPUs | {demanda_aplicacao[1]:.0f} GB RAM | {demanda_aplicacao[2]:.0f} GB SSD\n")

for rec in resultados_rec:
    specs = rec['specs']
    print(f"Ranking: {rec['posicao']}º lugar")
    print(f" Servidor Recomendado: {rec['nome']}")
    print(f" Configuração:        {specs[0]:.0f} vCPUs, {specs[1]:.0f} GB RAM, {specs[2]:.0f} GB SSD")
    print(f" Distância Geométrica: {rec['distancia']:.4f}")
    print("-" * 65)

DESAFIO 3: RELATÓRIO DE RECOMENDAÇÃO DE SERVIDORES CLOUD
Perfil Demanda Solicitada: 12 vCPUs | 28 GB RAM | 450 GB SSD

Ranking: 1º lugar
 Servidor Recomendado: Database Enterprise
 Configuração:        16 vCPUs, 32 GB RAM, 500 GB SSD
 Distância Geométrica: 50.3190
-----------------------------------------------------------------
Ranking: 2º lugar
 Servidor Recomendado: Medium Backend & Cache
 Configuração:        8 vCPUs, 16 GB RAM, 250 GB SSD
 Distância Geométrica: 200.3996
-----------------------------------------------------------------
